# Dataset Cleaning Notebook

This notebook cleans the dataset by removing corrupted files, fixing issues, and ensuring consistency.

## Objectives:
- Remove unreadable or corrupted images
- Remove or correct invalid bounding boxes
- Detect and remove duplicated images or labels
- Normalize file naming conventions
- Enforce strict one-to-one image–label mapping

In [ ]:
# Import required libraries
import os
import json
import shutil
from pathlib import Path
from PIL import Image
import hashlib
from collections import defaultdict, Counter
from tqdm import tqdm
import yaml
import numpy as np
from datetime import datetime

# Set paths
BASE_DIR = Path('AletrixGrad-dataset')
DATA_YAML = BASE_DIR / 'data.yaml'
CLEANED_DIR = Path('dataset_cleaned')
REPORTS_DIR = Path('reports')

# Load validation report if available
validation_report_path = REPORTS_DIR / 'validation_report.json'

print(f"Base directory: {BASE_DIR.absolute()}")
print(f"Cleaned dataset will be saved to: {CLEANED_DIR.absolute()}")
print(f"Validation report: {validation_report_path.absolute()}")

# Create cleaned directory structure
CLEANED_DIR.mkdir(exist_ok=True)

## Load Validation Report and Dataset Configuration

In [ ]:
# Load data.yaml
with open(DATA_YAML, 'r') as f:
    config = yaml.safe_load(f)

NUM_CLASSES = config['nc']
CLASS_NAMES = config['names']
SPLITS = ['train', 'valid', 'test']

print(f"Number of classes: {NUM_CLASSES}")
print(f"Class names: {CLASS_NAMES}")

# Load validation report
if validation_report_path.exists():
    with open(validation_report_path, 'r') as f:
        validation_report = json.load(f)
    print(f"\n✓ Validation report loaded from: {validation_report_path}")
    print(f"  Report timestamp: {validation_report.get('timestamp', 'N/A')}")
else:
    print(f"\n⚠ Warning: Validation report not found at {validation_report_path}")
    print("  Will perform cleaning without validation report (will re-validate during cleaning)")
    validation_report = None

## Cleaning Functions

In [ ]:
def compute_image_hash(image_path):
    """Compute MD5 hash of image file for duplicate detection."""
    hash_md5 = hashlib.md5()
    try:
        with open(image_path, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"  Warning: Could not hash {image_path.name}: {e}")
        return None

def fix_bbox_values(class_id, x_center, y_center, width, height, num_classes):
    """
    Fix bounding box values outside [0,1] range by clamping.
    Returns: (fixed_x, fixed_y, fixed_w, fixed_h, was_fixed)
    """
    was_fixed = False
    
    # Clamp class ID
    if class_id < 0 or class_id >= num_classes:
        return None, None, None, None, False  # Cannot fix invalid class ID
    
    # Clamp x_center, y_center to [0, 1]
    if x_center < 0:
        x_center = 0.0
        was_fixed = True
    elif x_center > 1:
        x_center = 1.0
        was_fixed = True
    
    if y_center < 0:
        y_center = 0.0
        was_fixed = True
    elif y_center > 1:
        y_center = 1.0
        was_fixed = True
    
    # Clamp width and height to [0, 1]
    if width < 0:
        width = 0.0
        was_fixed = True
    elif width > 1:
        width = 1.0
        was_fixed = True
    
    if height < 0:
        height = 0.0
        was_fixed = True
    elif height > 1:
        height = 1.0
        was_fixed = True
    
    # Adjust bbox if center + half_size exceeds 1 or center - half_size < 0
    if x_center + width/2 > 1:
        # Adjust x_center to keep bbox within bounds
        x_center = max(0, 1 - width/2)
        was_fixed = True
    if x_center - width/2 < 0:
        x_center = min(1, width/2)
        was_fixed = True
    
    if y_center + height/2 > 1:
        y_center = max(0, 1 - height/2)
        was_fixed = True
    if y_center - height/2 < 0:
        y_center = min(1, height/2)
        was_fixed = True
    
    # Remove boxes with zero or near-zero dimensions
    if width < 1e-6 or height < 1e-6:
        return None, None, None, None, False
    
    return x_center, y_center, width, height, was_fixed

def fix_label_file(label_path, num_classes, log_fn):
    """
    Fix invalid bounding boxes in a label file.
    Returns: (is_valid, num_fixed, num_removed)
    """
    if not label_path.exists():
        return False, 0, 0
    
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        fixed_lines = []
        num_fixed = 0
        num_removed = 0
        
        for line_num, line in enumerate(lines, 1):
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) != 5:
                num_removed += 1
                log_fn(f"    Line {line_num}: Removed - invalid format (expected 5 values, got {len(parts)})")
                continue
            
            try:
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
            except ValueError:
                num_removed += 1
                log_fn(f"    Line {line_num}: Removed - invalid numeric values")
                continue
            
            # Check class ID
            if class_id < 0 or class_id >= num_classes:
                num_removed += 1
                log_fn(f"    Line {line_num}: Removed - class ID {class_id} out of range")
                continue
            
            # Fix bbox values
            fixed_x, fixed_y, fixed_w, fixed_h, was_fixed = fix_bbox_values(
                class_id, x_center, y_center, width, height, num_classes
            )
            
            if fixed_x is None:
                num_removed += 1
                log_fn(f"    Line {line_num}: Removed - bbox cannot be fixed")
                continue
            
            if was_fixed:
                num_fixed += 1
                log_fn(f"    Line {line_num}: Fixed bbox values")
            
            fixed_lines.append(f"{class_id} {fixed_x:.10f} {fixed_y:.10f} {fixed_w:.10f} {fixed_h:.10f}\n")
        
        # Write fixed labels back
        if fixed_lines:
            with open(label_path, 'w') as f:
                f.writelines(fixed_lines)
            return True, num_fixed, num_removed
        else:
            # Empty file after fixing - will be removed later
            return False, num_fixed, num_removed
        
    except Exception as e:
        log_fn(f"  Error fixing label {label_path.name}: {e}")
        return False, 0, 0

## Initialize Cleaning Logs and Statistics

In [ ]:
# Initialize cleaning logs and statistics
cleaning_log = []
cleaning_stats = {
    'splits': {},
    'summary': {
        'original_images': 0,
        'original_labels': 0,
        'cleaned_images': 0,
        'cleaned_labels': 0,
        'removed_corrupted_images': 0,
        'removed_missing_labels': 0,
        'removed_invalid_labels': 0,
        'removed_empty_labels': 0,
        'removed_orphaned_labels': 0,
        'removed_duplicates': 0,
        'fixed_bboxes': 0,
        'removed_bboxes': 0
    }
}

def log_action(message):
    """Log an action message."""
    cleaning_log.append(message)
    print(message)

## Process Each Split

In [ ]:
# Process each split
for split in SPLITS:
    log_action(f"\n{'='*80}")
    log_action(f"Processing {split.upper()} split...")
    log_action(f"{'='*80}")
    
    split_dir = BASE_DIR / split
    images_dir = split_dir / 'images'
    labels_dir = split_dir / 'labels'
    
    if not images_dir.exists() or not labels_dir.exists():
        log_action(f"  ⚠ Warning: {split} directory structure not found, skipping...")
        continue
    
    # Create cleaned split directories
    cleaned_split_dir = CLEANED_DIR / split
    cleaned_images_dir = cleaned_split_dir / 'images'
    cleaned_labels_dir = cleaned_split_dir / 'labels'
    cleaned_images_dir.mkdir(parents=True, exist_ok=True)
    cleaned_labels_dir.mkdir(parents=True, exist_ok=True)
    
    split_stats = {
        'original_images': 0,
        'original_labels': 0,
        'cleaned_images': 0,
        'cleaned_labels': 0,
        'removed_corrupted_images': 0,
        'removed_missing_labels': 0,
        'removed_invalid_labels': 0,
        'removed_empty_labels': 0,
        'removed_orphaned_labels': 0,
        'removed_duplicates': 0,
        'fixed_bboxes': 0,
        'removed_bboxes': 0
    }
    
    # Get validation report data for this split if available
    split_validation = None
    if validation_report and split in validation_report.get('splits', {}):
        split_validation = validation_report['splits'][split]
        corrupted_paths = {Path(item['path']) for item in split_validation.get('corrupted_images', [])}
        missing_label_paths = {Path(p) for p in split_validation.get('missing_labels', [])}
        invalid_label_paths = {Path(item['path']) for item in split_validation.get('invalid_labels', [])}
        orphaned_label_paths = {Path(p) for p in split_validation.get('orphaned_labels', [])}
    else:
        corrupted_paths = set()
        missing_label_paths = set()
        invalid_label_paths = set()
        orphaned_label_paths = set()
    
    # Get all image files
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_files = []
    for ext in image_extensions:
        image_files.extend(list(images_dir.glob(f'*{ext}')))
        image_files.extend(list(images_dir.glob(f'*{ext.upper()}')))
    
    label_files = list(labels_dir.glob('*.txt'))
    
    split_stats['original_images'] = len(image_files)
    split_stats['original_labels'] = len(label_files)
    
    log_action(f"  Original: {len(image_files)} images, {len(label_files)} labels")
    
    # Step 1: Identify files to keep (valid image-label pairs)
    valid_pairs = {}  # {img_path: (img_path, label_path)}
    image_hashes = {}  # {hash: img_path} for duplicate detection
    
    log_action(f"\n  Step 1: Identifying valid image-label pairs...")
    
    for img_path in tqdm(image_files, desc=f"  Scanning {split} images"):
        img_name = img_path.stem
        
        # Check if image is corrupted
        if img_path in corrupted_paths or not img_path.exists():
            split_stats['removed_corrupted_images'] += 1
            cleaning_stats['summary']['removed_corrupted_images'] += 1
            log_action(f"    ✗ Removed corrupted image: {img_path.name}")
            continue
        
        # Check if image is readable
        try:
            img = Image.open(img_path)
            img.verify()
            img.close()
        except Exception:
            split_stats['removed_corrupted_images'] += 1
            cleaning_stats['summary']['removed_corrupted_images'] += 1
            log_action(f"    ✗ Removed unreadable image: {img_path.name}")
            continue
        
        # Check for corresponding label
        label_path = labels_dir / f'{img_name}.txt'
        if not label_path.exists() or img_path in missing_label_paths:
            split_stats['removed_missing_labels'] += 1
            cleaning_stats['summary']['removed_missing_labels'] += 1
            log_action(f"    ✗ Removed image with missing label: {img_path.name}")
            continue
        
        # Check for duplicate images (hash-based)
        img_hash = compute_image_hash(img_path)
        if img_hash is None:
            continue  # Skip if hash computation failed
        
        if img_hash in image_hashes:
            # Duplicate found
            split_stats['removed_duplicates'] += 1
            cleaning_stats['summary']['removed_duplicates'] += 1
            log_action(f"    ✗ Removed duplicate image: {img_path.name} (duplicate of {image_hashes[img_hash].name})")
            continue
        
        image_hashes[img_hash] = img_path
        valid_pairs[img_path] = (img_path, label_path)
    
    log_action(f"  Found {len(valid_pairs)} valid image-label pairs")
    
    # Step 2: Fix invalid labels and remove empty ones
    log_action(f"\n  Step 2: Fixing invalid labels...")
    
    labels_to_fix = []
    for img_path, (_, label_path) in valid_pairs.items():
        if label_path in invalid_label_paths or not label_path.exists():
            labels_to_fix.append((img_path, label_path))
    
    for img_path, label_path in tqdm(labels_to_fix, desc=f"  Fixing {split} labels"):
        def log_label(msg):
            log_action(f"    {label_path.name}: {msg}")
        
        is_valid, num_fixed, num_removed = fix_label_file(label_path, NUM_CLASSES, log_label)
        
        split_stats['fixed_bboxes'] += num_fixed
        split_stats['removed_bboxes'] += num_removed
        cleaning_stats['summary']['fixed_bboxes'] += num_fixed
        cleaning_stats['summary']['removed_bboxes'] += num_removed
        
        if not is_valid:
            # Label is now empty or still invalid - remove the pair
            if img_path in valid_pairs:
                del valid_pairs[img_path]
                split_stats['removed_invalid_labels'] += 1
                cleaning_stats['summary']['removed_invalid_labels'] += 1
                log_action(f"    ✗ Removed pair with empty/invalid label: {img_path.name}")
    
    # Step 3: Check for orphaned labels and empty label files
    log_action(f"\n  Step 3: Checking for orphaned and empty labels...")
    
    for label_path in tqdm(label_files, desc=f"  Checking {split} labels"):
        label_name = label_path.stem
        
        # Check if label is orphaned
        img_found = False
        for ext in image_extensions:
            if (images_dir / f'{label_name}{ext}').exists() or \
               (images_dir / f'{label_name}{ext.upper()}').exists():
                img_found = True
                break
        
        if not img_found or label_path in orphaned_label_paths:
            # Note: We don't delete from original dataset, just skip copying
            split_stats['removed_orphaned_labels'] += 1
            cleaning_stats['summary']['removed_orphaned_labels'] += 1
            log_action(f"    ✗ Skipped orphaned label (not copied): {label_path.name}")
            continue
        
        # Check if label file is empty
        try:
            with open(label_path, 'r') as f:
                content = f.read().strip()
                if not content:
                    # Note: We don't delete from original dataset, just skip copying
                    # Also remove corresponding image from valid_pairs if it exists
                    for img_path in list(valid_pairs.keys()):
                        if img_path.stem == label_name:
                            del valid_pairs[img_path]
                            split_stats['removed_empty_labels'] += 1
                            cleaning_stats['summary']['removed_empty_labels'] += 1
                            log_action(f"    ✗ Removed pair with empty label: {img_path.name}")
                            break
                    split_stats['removed_empty_labels'] += 1
                    cleaning_stats['summary']['removed_empty_labels'] += 1
                    log_action(f"    ✗ Removed empty label: {label_path.name}")
        except Exception:
            pass
    
    # Step 4: Copy valid pairs to cleaned directory with normalized naming
    log_action(f"\n  Step 4: Copying cleaned files with normalized naming...")
    
    for idx, (img_path, (_, label_path)) in enumerate(tqdm(valid_pairs.items(), desc=f"  Copying {split} files"), 1):
        # Normalize filename: use split prefix + sequential number
        img_ext = img_path.suffix.lower()
        if img_ext not in ['.jpg', '.jpeg']:
            img_ext = '.jpg'  # Normalize to .jpg
        
        new_img_name = f"{split}_{idx:06d}{img_ext}"
        new_label_name = f"{split}_{idx:06d}.txt"
        
        new_img_path = cleaned_images_dir / new_img_name
        new_label_path = cleaned_labels_dir / new_label_name
        
        # Copy image
        try:
            shutil.copy2(img_path, new_img_path)
        except Exception as e:
            log_action(f"    ✗ Error copying image {img_path.name}: {e}")
            continue
        
        # Copy and verify label
        try:
            shutil.copy2(label_path, new_label_path)
            split_stats['cleaned_images'] += 1
            split_stats['cleaned_labels'] += 1
        except Exception as e:
            log_action(f"    ✗ Error copying label {label_path.name}: {e}")
            if new_img_path.exists():
                new_img_path.unlink()
            continue
    
    cleaning_stats['summary']['original_images'] += split_stats['original_images']
    cleaning_stats['summary']['original_labels'] += split_stats['original_labels']
    cleaning_stats['summary']['cleaned_images'] += split_stats['cleaned_images']
    cleaning_stats['summary']['cleaned_labels'] += split_stats['cleaned_labels']
    
    cleaning_stats['splits'][split] = split_stats
    
    log_action(f"\n  {split.upper()} Cleaning Summary:")
    log_action(f"    Original: {split_stats['original_images']} images, {split_stats['original_labels']} labels")
    log_action(f"    Cleaned: {split_stats['cleaned_images']} images, {split_stats['cleaned_labels']} labels")
    log_action(f"    Removed - Corrupted images: {split_stats['removed_corrupted_images']}")
    log_action(f"    Removed - Missing labels: {split_stats['removed_missing_labels']}")
    log_action(f"    Removed - Invalid labels: {split_stats['removed_invalid_labels']}")
    log_action(f"    Removed - Empty labels: {split_stats['removed_empty_labels']}")
    log_action(f"    Removed - Orphaned labels: {split_stats['removed_orphaned_labels']}")
    log_action(f"    Removed - Duplicates: {split_stats['removed_duplicates']}")
    log_action(f"    Fixed bboxes: {split_stats['fixed_bboxes']}")
    log_action(f"    Removed bboxes: {split_stats['removed_bboxes']}")

## Copy data.yaml to Cleaned Dataset

In [ ]:
# Copy data.yaml to cleaned dataset
cleaned_data_yaml = CLEANED_DIR / 'data.yaml'
shutil.copy2(DATA_YAML, cleaned_data_yaml)

# Update paths in data.yaml for cleaned dataset
with open(cleaned_data_yaml, 'r') as f:
    cleaned_config = yaml.safe_load(f)

# Update paths to be relative to cleaned directory
cleaned_config['path'] = str(CLEANED_DIR.absolute())
cleaned_config['train'] = 'train/images'
cleaned_config['val'] = 'valid/images'
cleaned_config['test'] = 'test/images'

with open(cleaned_data_yaml, 'w') as f:
    yaml.dump(cleaned_config, f, default_flow_style=False, sort_keys=False)

log_action(f"\n✓ Updated data.yaml in cleaned dataset: {cleaned_data_yaml.absolute()}")

## Generate Cleaning Summary Report

In [ ]:
# Print comprehensive summary
log_action(f"\n{'='*80}")
log_action("DATASET CLEANING SUMMARY")
log_action(f"{'='*80}")

summary = cleaning_stats['summary']

log_action(f"\nOverall Statistics:")
log_action(f"  Original Images: {summary['original_images']}")
log_action(f"  Original Labels: {summary['original_labels']}")
log_action(f"  Cleaned Images: {summary['cleaned_images']}")
log_action(f"  Cleaned Labels: {summary['cleaned_labels']}")

log_action(f"\nRemoved Items:")
log_action(f"  Corrupted Images: {summary['removed_corrupted_images']}")
log_action(f"  Missing Labels: {summary['removed_missing_labels']}")
log_action(f"  Invalid Labels: {summary['removed_invalid_labels']}")
log_action(f"  Empty Labels: {summary['removed_empty_labels']}")
log_action(f"  Orphaned Labels: {summary['removed_orphaned_labels']}")
log_action(f"  Duplicates: {summary['removed_duplicates']}")

log_action(f"\nFixed Items:")
log_action(f"  Fixed Bounding Boxes: {summary['fixed_bboxes']}")
log_action(f"  Removed Invalid Bounding Boxes: {summary['removed_bboxes']}")

# Calculate retention rates
if summary['original_images'] > 0:
    retention_rate = (summary['cleaned_images'] / summary['original_images']) * 100
    log_action(f"\nRetention Rate: {retention_rate:.2f}% ({summary['cleaned_images']}/{summary['original_images']})")

# Per-split summary
log_action(f"\n{'='*80}")
log_action("PER-SPLIT SUMMARY")
log_action(f"{'='*80}")

for split in SPLITS:
    if split in cleaning_stats['splits']:
        split_stats = cleaning_stats['splits'][split]
        log_action(f"\n{split.upper()}:")
        log_action(f"  Original: {split_stats['original_images']} images")
        log_action(f"  Cleaned: {split_stats['cleaned_images']} images")
        if split_stats['original_images'] > 0:
            split_retention = (split_stats['cleaned_images'] / split_stats['original_images']) * 100
            log_action(f"  Retention: {split_retention:.2f}%")

## Save Cleaning Report

In [ ]:
# Save cleaning report to JSON
cleaning_report = {
    'timestamp': datetime.now().isoformat(),
    'source_dataset': str(BASE_DIR.absolute()),
    'cleaned_dataset': str(CLEANED_DIR.absolute()),
    'validation_report_used': str(validation_report_path) if validation_report_path.exists() else None,
    'config': {
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES
    },
    'statistics': cleaning_stats,
    'log': cleaning_log
}

# Convert Path objects to strings
def convert_paths_for_json(obj):
    if isinstance(obj, dict):
        return {k: convert_paths_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_paths_for_json(item) for item in obj]
    elif isinstance(obj, Path):
        return str(obj)
    return obj

cleaning_report_json = convert_paths_for_json(cleaning_report)

report_path = REPORTS_DIR / 'cleaning_report.json'
REPORTS_DIR.mkdir(exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(cleaning_report_json, f, indent=2)

log_action(f"\n✓ Cleaning report saved to: {report_path.absolute()}")

# Save cleaning log to text file
log_path = REPORTS_DIR / 'cleaning_log.txt'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(cleaning_log))

log_action(f"✓ Cleaning log saved to: {log_path.absolute()}")

log_action(f"\n{'='*80}")
log_action("CLEANING COMPLETE!")
log_action(f"{'='*80}")
log_action(f"\nCleaned dataset saved to: {CLEANED_DIR.absolute()}")
log_action(f"Original dataset preserved at: {BASE_DIR.absolute()}")
log_action(f"\nNext steps:")
log_action(f"  1. Review cleaning report: {report_path}")
log_action(f"  2. Verify cleaned dataset in: {CLEANED_DIR}")
log_action(f"  3. Proceed to Notebook 03: Dataset Balancing & Augmentation")